# V11 bilingual notice checker

V11 compares the information conveyed in both directions. Its frozen equivalence classifier uses contradiction, entailment, neutral, similarity and explicit-fact signals. A corroborated qualifier check helps identify missing conditions. No V7 fallback is called.

The score is a decision score, not a calibrated probability. A clean verdict means no discrepancy was found. The supplied report describes the fresh benchmark and its limitations.

The included source and pretrained model revisions are frozen. This notebook is a Colab-ready copy; the reported benchmark was executed in the local CPU runtime, not Colab. The NLLB component is a research model licensed CC-BY-NC-4.0. See https://huggingface.co/facebook/nllb-200-distilled-600M.


This copy adds a Gradio demo app (after the single example) and puts the checker folder on `sys.path` so the loader runs in Colab. The frozen checker files and their checksums are unchanged.

In [ ]:
!pip -q install sentence-transformers==5.7.0 transformers==5.22.1 sentencepiece==0.2.3 opencc-python-reimplemented==0.1.7 pandas scipy numpy gradio


## The checker's code, file by file
V11 is a small Python package of 11 files. Each cell below writes one file to `/content/v11_checker/` **exactly as it
was frozen**, so you can read every line. The notes above each cell explain what the file does. The code itself is not
edited, because the last cell checks every file against the SHA-256 checksum recorded when V11 was frozen: changing a
single character would make that check fail.

**Order:** facts → terminology → models and signals → features → missing-detail rule → decision → trained weights →
benchmark runner → settings.

In [ ]:
from pathlib import Path
folder = Path('/content/v11_checker'); folder.mkdir(exist_ok=True)
def save(name, text):
    (folder / name).write_text(text, encoding='utf-8')   # writes the file byte for byte
    print(f'wrote {name}: {len(text.encode())} bytes')

### `facts.py`: hand-written fact checks
Finds small, clear facts in each language and compares them. A fact only counts when it appears **exactly once** on each
side, so the rules never guess which of several values is meant.
- `normal()` tidies the text (full-width characters, spacing) so both versions are compared fairly.
- `number()`, `CN`, `NU`: read numbers written as digits or as Chinese numerals (三百 → 300).
- `CONCEPTS` and `concept_facts()`: one-word facts in both languages: colour, language, page orientation, material,
  file format, weekday, frequency, original vs copy.
- `UNITS`, `MONTHS` and `quantity_facts()`: amounts with units (HK$, kg, cm, days) and dates.
- `direct_evidence()`: the main entry point. Compares the facts above, plus email addresses, web links, reference codes
  and clock times, and returns each mismatch with the words found in each language.
- `boundary_evidence()`: inclusive vs exclusive limits ("aged 65 or above" vs "over 65").
- `ACTIONS` and `modality_evidence()`: must / may / need not, for actions such as bring, submit, pay.
- `clock_facts()`: clock times in both languages (下午2時 = 14:00).

In [ ]:
save('facts.py', r'''"""Small, context-independent bilingual facts. Never compare bare negation counts.

Only a uniquely resolved property in BOTH passages can establish a conflict.
These are checks, not a claim of exhaustive linguistic coverage.
"""
import re
import unicodedata
from collections import defaultdict
from decimal import Decimal


def normal(text):
    return unicodedata.normalize('NFKC', text).replace('’', "'").strip()

CN = dict(zip('零〇○一二兩三四五六七八九', [0,0,0,1,2,2,3,4,5,6,7,8,9]))
NU = {'十':10, '百':100, '千':1000, '萬':10000}
NUM = r'(?:\d+(?:[,.]\d+)*|[零〇○一二兩三四五六七八九十百千萬]+)'
ENNUM = dict(zip('zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty'.split(),range(21)))

def number(s):
    s=s.lower().replace(',','')
    if s in ENNUM:return Decimal(ENNUM[s])
    if re.fullmatch(r'\d+(?:\.\d+)?',s):return Decimal(s)
    if all(x in CN for x in s):return Decimal(''.join(str(CN[x]) for x in s))
    total=cur=section=0
    for x in s:
        if x in CN:cur=CN[x]
        elif x in NU:
            if NU[x]==10000:total+=(section+cur)*10000;section=cur=0
            else:section+=(cur or 1)*NU[x];cur=0
        else:raise ValueError(s)
    return Decimal(total+section+cur)

# Closed-value concepts; word-level boundaries avoid e.g. 'red' inside 'required'.
CONCEPTS = {
 'colour': [('red',r'red',r'紅色'),('blue',r'blue',r'藍色'),('green',r'green',r'綠色'),('yellow',r'yellow',r'黃色'),('black',r'black',r'黑色'),('white',r'white',r'白色'),('orange',r'orange',r'橙色'),('purple',r'purple',r'紫色')],
 'language': [('English',r'English',r'英文|英語'),('Chinese',r'Chinese',r'中文|漢語'),('Cantonese',r'Cantonese',r'粵語|廣東話'),('Mandarin',r'Mandarin|Putonghua',r'普通話|國語'),('Japanese',r'Japanese',r'日文|日語'),('Korean',r'Korean',r'韓文|韓語'),('French',r'French',r'法文|法語'),('Spanish',r'Spanish',r'西班牙文|西班牙語')],
 'orientation': [('portrait',r'portrait',r'直向|縱向'),('landscape',r'landscape',r'橫向')],
 'material': [('glass',r'glass',r'玻璃'),('plastic',r'plastic',r'塑膠|塑料|膠樽'),('metal',r'metal',r'金屬'),('aluminium',r'alumin(?:ium|um)',r'鋁'),('wood',r'wood(?:en)?',r'木製|木材'),('paper',r'paper',r'紙製|紙張')],
 'file format': [(x, x, x) for x in ['PDF','CSV','DOCX','XLSX','JPEG','PNG','ZIP']],
 'weekday': [(e,e+r's?',r'(?:星期|週|周)'+z) for e,z in zip(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],['一','二','三','四','五','六','[日天]'])],
 'frequency': [('daily',r'daily|every day',r'每日|每天'),('weekly',r'weekly|every week',r'每週|每星期|每周'),('monthly',r'monthly|every month',r'每月'),('quarterly',r'quarterly|every three months',r'每季|每三個月'),('yearly',r'yearly|annually|annual|every year',r'每年|一年一度')],
 'document form': [('original',r'original(?:s)?',r'正本|原件'),('copy',r'cop(?:y|ies)|photocop(?:y|ies)',r'副本|影印本')],
}

def evidence(category, z, e):
    return dict(category=category,chinese_value=str(z[0]),english_value=str(e[0]),chinese_span=z[1],english_span=e[1])

def unique(hits):
    values={x[0] for x in hits}
    return hits[0] if len(values)==1 else None


def concept_facts(text, lang):
    found={}
    for category, entries in CONCEPTS.items():
        hits=[]
        for value,ep,zp in entries:
            pat=r'\b(?:'+ep+r')\b' if lang=='en' else zp
            hits.extend((value,m[0]) for m in re.finditer(pat,text,re.I))
        f=unique(hits)
        if f:found[category]=f
    return found

# Quantity units include a dimension and conversion multiplier. Correct changes
# such as 1 m / 100 cm are equivalent; percentages in 百分比 are not the number 100.
UNITS=[
 ('length','1',r'm(?:etre|eter)s?|m',r'米|公尺'),
 ('length','.01',r'centim(?:etre|eter)s?|cm',r'厘米|公分'),
 ('length','.001',r'millim(?:etre|eter)s?|mm',r'毫米'),
 ('length','1000',r'kilom(?:etre|eter)s?|km',r'公里|千米'),
 ('mass','1',r'kilograms?|kg',r'公斤|千克'),('mass','.001',r'grams?|g',r'克|公克'),
 ('duration','1',r'minutes?|mins?',r'分鐘'),('duration','60',r'hours?|hrs?',r'小時'),
 ('days','1',r'days?',r'天|日'),('weeks','1',r'weeks?',r'星期|週|周'),
 ('age','1',r'years? old|years? of age',r'歲'),
 ('amount HKD','1',r'HKD|HK dollars?|Hong Kong dollars?',r'港元'),
 ('percentage','1',r'%|percent|per cent',r'%|巴仙'),
 ('pages','1',r'pages?',r'頁'),('floors','1',r'floors?',r'樓'),
 ('occurrences','1',r'times?',r'次'),
 ('count','1',r'people|persons?|pupils?|students?|participants?|copies|items?|forms?',r'人|名|個|份'),
]
MONTHS={x:i+1 for i,x in enumerate('January February March April May June July August September October November December'.split())}

def quantity_facts(text, lang):
    result=defaultdict(list)
    t=normal(text)
    num=NUM if lang=='zh' else r'(?:\d+(?:[,.]\d+)*|'+ '|'.join(ENNUM)+r')'
    for dimension,mult,ep,zp in UNITS:
        pat=(rf'({num})\s*(?:{zp})(?![a-z])' if lang=='zh' else rf'\b({num})\s*(?:{ep})(?![a-z])')
        for m in re.finditer(pat,t,re.I):
            # 二次 / 一般 / 百分比 cannot be mistaken for numeric quantities.
            result[dimension].append((str(number(m[1])*Decimal(mult)),m[0]))
    if lang=='en':
        for m in re.finditer(r'\bHK\$\s*(\d+(?:[,.]\d+)*)',t,re.I):result['amount HKD'].append((str(number(m[1])),m[0]))
        for m in re.finditer(r'\b(\d{1,2})(?:st|nd|rd|th)?\s+('+'|'.join(MONTHS)+r')\b',t,re.I):result['date'].append((f'{MONTHS[m[2].title()]}-{int(m[1])}',m[0]))
        for m in re.finditer(r'\b('+'|'.join(MONTHS)+r')\s+(\d{1,2})(?:st|nd|rd|th)?\b',t,re.I):result['date'].append((f'{MONTHS[m[1].title()]}-{int(m[2])}',m[0]))
    else:
        for m in re.finditer(rf'({NUM})月({NUM})[日號]',t):result['date'].append((f'{number(m[1])}-{number(m[2])}',m[0]))
        # Dates are not durations. A date's 日 is removed from day quantities.
        result['days']=[q for q in result['days'] if not re.search(r'月'+re.escape(q[1]),t)]
    # Decimals are canonicalised for equality after SI conversion.
    for category, hits in result.items():
        if category!='date':result[category]=[(str(Decimal(v).normalize()),s) for v,s in hits]
    return dict(result)


def direct_evidence(chinese, english):
    z,e=normal(chinese),normal(english)
    out=[]
    for extractor in (concept_facts,quantity_facts):
        zf,ef=extractor(z,'zh'),extractor(e,'en')
        for category in sorted(zf.keys() & ef.keys()):
            a,b=zf[category],ef[category]
            if extractor==quantity_facts:
                # Sets of values cannot assign semantic roles; defer repeated
                # distinct quantities to contextual inference rather than guess.
                a,b=unique(a),unique(b)
            if a and b and a[0]!=b[0]:out.append(evidence(category,a,b))
    for category,pattern in [
        ('email address',r'[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}'),
        ('URL destination',r'https?://[^\s<>，。；）」]+'),
        ('reference identifier',r'(?<![A-Za-z\d])[A-Z]{1,5}[-/]\d{2,}(?:[-/]\d+)?')]:
        zs=[(m[0].rstrip('.,;)'),m[0]) for m in re.finditer(pattern,z)]
        es=[(m[0].rstrip('.,;)'),m[0]) for m in re.finditer(pattern,e)]
        a,b=unique(zs),unique(es)
        if a and b and a[0]!=b[0]:out.append(evidence(category,a,b))
    zc,ec=clock_facts(z,'zh'),clock_facts(e,'en')
    a,b=unique(zc),unique(ec)
    if a and b and a[0]!=b[0]:out.append(evidence('clock time',a,b))
    out.extend(boundary_evidence(z,e))
    out.extend(modality_evidence(z,e))
    return out


def boundary_evidence(z,e):
    out=[]
    # Position words only when used spatially, not the past tense of leave.
    en_directions=[(v,r'(?:(?:to|towards?|points?|pointing|facing|turn)\s+(?:the\s+)?'+pat+r'|'+pat+r'[- ]hand)') for v,pat in [('left','left'),('right','right'),('up','upwards?|up'),('down','downwards?|down')]]
    zh_directions=[('left',r'左(?:方|側|邊)'),('right',r'右(?:方|側|邊)'),('up',r'上(?:方|側|邊)'),('down',r'下(?:方|側|邊)')]
    zh=unique([(v,m[0]) for v,p in zh_directions for m in re.finditer(p,z)])
    en=unique([(v,m[0]) for v,p in en_directions for m in re.finditer(p,e,re.I)])
    if zh and en and zh[0]!=en[0]:out.append(evidence('spatial direction',zh,en))
    # Unit-only instructions can change even when there is no numeric value.
    for dim in ['length','mass']:
        zs=[];es=[]
        for d,mult,ep,zp in UNITS:
            if d!=dim:continue
            zs.extend((mult,m[0]) for m in re.finditer(r'(?:以|用|單位[為是:：])(?:'+zp+r')',z))
            es.extend((mult,m[0]) for m in re.finditer(r'\b(?:in|using)\s+(?:'+ep+r')\b',e,re.I))
        a,b=unique(zs),unique(es)
        if a and b and a[0]!=b[0]:out.append(evidence(dim+' unit',a,b))
    # Inclusive / exclusive boundaries must refer to the same explicit value.
    zn=NUM;en=r'\d+(?:\.\d+)?'
    zhpatterns=[('>=',rf'不少於({zn})|最少({zn})|({zn})(?:歲)?或以上'),('>',rf'超過({zn})|高於({zn})'),('<=',rf'不多於({zn})|最多({zn})|({zn})(?:歲)?或以下'),('<',rf'少於({zn})|未滿({zn})|低於({zn})')]
    enpatterns=[('>=',rf'\bat least ({en})|\b({en})(?: years?)? or (?:more|over|above)'),('>',rf'\bmore than ({en})|\babove ({en})'),('<=',rf'\bat most ({en})|\bno more than ({en})|\b({en})(?: years?)? or (?:less|under|below)'),('<',rf'\bless than ({en})|\bunder ({en})|\bbelow ({en})')]
    def limits(ps,t):
        found=[]
        for op,pat in ps:
            for m in re.finditer(pat,t,re.I):found.append(((op,str(number(next(g for g in m.groups() if g)))),m[0]))
        return unique(found)
    a,b=limits(zhpatterns,z),limits(enpatterns,e)
    if a and b and a[0][1]==b[0][1] and a[0][0]!=b[0][0]:out.append(evidence('inclusive/exclusive boundary',a,b))
    return out

# Optional versus required is material even when neither side is a logical
# contradiction. Only attach an explicit modal immediately before a shared action.
ACTIONS = {
 'submit': (r'submit|lodge|hand in',r'提交|遞交|交回'),
 'bring': (r'bring',r'攜帶|攜同|帶備|帶同'),
 'show': (r'show|present|produce',r'出示'),
 'pay': (r'pay',r'繳付|支付|繳交'),
 'register': (r'register|enrol|enroll',r'登記|報名'),
 'attend': (r'attend',r'出席'),
 'upload': (r'upload',r'上載|上傳'),
 'sign': (r'sign',r'簽署'),
 'retain': (r'keep|retain',r'保留'),
 'collect': (r'collect',r'領取|取件'),
 'reply': (r'reply|respond',r'回覆|回復'),
 'send': (r'send|forward',r'發送|寄送'),
 'use': (r'use',r'使用'),
 'enter': (r'enter',r'進入'),
 'cancel': (r'cancel',r'取消'),
 'replace': (r'replace',r'更換|取代'),
}

def modality_evidence(z,e):
    # Unresolved conditional scope is left to contextual inference, not reduced
    # to a document-wide bag of obligation tokens.
    if re.search(r'\b(?:if|unless|only|except|whether)\b',e,re.I) or re.search(r'如|若|除非|只有|僅|才|除外',z):return []
    out=[]
    em=[('optional',r'(?:need not|do(?:es)? not (?:need|have) to|(?:is|are) not required to)\s*$'),
        ('prohibited',r'(?:must not|may not|cannot|can\x27t|(?:is|are) not allowed to)\s*$'),
        ('required',r'(?<!not )(?:must|need to|have to|has to|(?:is|are) required to)\s*$'),
        ('permitted',r'(?:may|can|(?:is|are) allowed to)\s*$')]
    zm=[('optional',r'(?:無須|毋須|不必|不用)\s*$'),('prohibited',r'(?:不得|不可|禁止|不准)\s*$'),
        ('required',r'(?<![毋無不])(?:必須|須)\s*$'),('permitted',r'(?<![不])(?:可以|可)\s*$')]
    def fact(t,ap,patterns,lang):
        actions=list(re.finditer(r'\b(?:'+ap+r')\b' if lang=='en' else ap,t,re.I))
        if len(actions)!=1:return None
        a=actions[0];prefix=t[max(0,a.start()-60):a.start()]
        for value,pat in patterns:
            m=re.search(pat,prefix,re.I)
            if m:return value,m[0]+a[0]
        return None
    for name,(ep,zp) in ACTIONS.items():
        a,b=fact(z,zp,zm,'zh'),fact(e,ep,em,'en')
        # Optional vs permitted can be equivalent; neither establishes an obligation.
        if a and b and a[0]!=b[0] and not {a[0],b[0]}<={'optional','permitted'}:
            out.append(evidence('obligation to '+name,a,b))
    return out


def clock_facts(text,lang):
    found=[];occupied=[]
    if lang=='en':
        for m in re.finditer(r'\b(\d{1,2})(?:[:.](\d{2}))?\s*([ap])\.?m\.?',text,re.I):
            h,mn=int(m[1]),int(m[2] or 0)
            if 1<=h<=12 and mn<60:
                found.append((str((h%12+(12 if m[3].lower()=='p' else 0))*60+mn),m[0]));occupied.append(m.span())
        for val,p in [(720,r'\bnoon\b'),(0,r'\bmidnight\b')]:
            found.extend((str(val),m[0]) for m in re.finditer(p,text,re.I))
    else:
        for m in re.finditer(rf'(上午|下午|晚上|凌晨|中午)({NUM})(?:時|點|:)(?:({NUM})分?)?',text):
            h,mn=int(number(m[2])),int(number(m[3])) if m[3] else 0
            if 0<=h<=12 and mn<60:
                found.append((str((h%12+(12 if m[1] in ['下午','晚上','中午'] else 0))*60+mn),m[0]));occupied.append(m.span())
        for val,p in [(720,r'正午'),(0,r'午夜')]:found.extend((str(val),m[0]) for m in re.finditer(p,text))
    for m in re.finditer(r'(?<!\d)(\d{1,2}):(\d{2})(?!\d)',text):
        if any(m.start()<b and m.end()>a for a,b in occupied):continue
        h,mn=int(m[1]),int(m[2])
        if h<24 and mn<60:found.append((str(h*60+mn),m[0]))
    return found
''')

### `terminology.py`: glossary for the translation model
Before the Chinese is machine-translated, a few rare terms (tree names, units, 業主立案法團, 正本/影印本) are replaced by
their English names so a mistranslated rare word does not look like a disagreement. The text is then converted from
Traditional to Simplified Chinese, which the translation model handles better.

In [ ]:
save('terminology.py', r'''"""Bounded terminology for translation cross-checks, independent of notice wording.

This is a glossary, not an equivalence verdict or a supported-notice whitelist.
Terms not listed continue through normal contextual translation. No benchmark
identifiers, target wording, labels or gold rationales enter this step.
"""
import re
from opencc import OpenCC
_TO_SIMPLIFIED=OpenCC("t2s")

TERMS = {
    '榕樹':'banyan tree', '樟樹':'camphor tree', '洋紫荊':'orchid tree',
    '相思樹':'acacia tree', '松樹':'pine tree', '橡樹':'oak tree',
    '楓樹':'maple tree', '棕櫚樹':'palm tree', '木棉樹':'cotton tree',
    '柳樹':'willow tree', '銀杏樹':'ginkgo tree',
    '裝裱':'mounting artwork',
    '直向格式':'portrait format', '縱向格式':'portrait format',
    '橫向格式':'landscape format',
    '毫米':'millimetres', '厘米':'centimetres', '公斤':'kilograms',
    '港元':'Hong Kong dollars', '工作天':'working days', '曆日':'calendar days',
    '正本':'original document', '影印本':'photocopy',
    '業主立案法團':"owners' corporation", '業主法團':"owners' corporation",
    '非牟利':'nonprofit', '私隱':'privacy', '電郵':'email',
}
_PATTERN=re.compile('|'.join(map(re.escape,sorted(TERMS,key=len,reverse=True))))

def translation_source(chinese):
    # Anchor known technical terms before generation, so a mistranslated rare
    # word does not masquerade as disagreement with the supplied English.
    return _TO_SIMPLIFIED.convert(_PATTERN.sub(lambda m:' '+TERMS[m[0]]+' ',chinese))
''')

### `backbone.py`: the three pretrained models and the signals
- `MODEL_IDS`, `MODEL_REVISIONS`: the exact model versions, pinned so results can be reproduced.
- `sentences()` and `aligned_groups()`: split each version into sentences and match Chinese to English passages with
  LaBSE, one-to-one.
- `SignalExtractor.__init__`: loads LaBSE, mDeBERTa NLI and NLLB-200 onto the GPU, with a cache so repeated runs are fast.
- `translate()`: NLLB-200 translates Chinese sentences into English (beam search, batches of 8). The translation is a
  cross-check, never the answer key.
- `nli_scores()`: mDeBERTa scores each sentence pair as contradiction / entailment / neutral (batches of 12).
- `predict_many()`: for each notice, rejects inputs that are empty or too long to check fully (these are marked for
  manual review rather than silently cut), then measures, for the whole notice and for each matched passage:
  cross-language NLI in both directions, NLI between the English and the machine translation, LaBSE similarity,
  translation quality, and the direct facts from `facts.py`.

In [ ]:
save('backbone.py', r'''"""V11: direct contextual inference and bilingual facts, with no V7 dependency.

The Chinese-to-English translation is a cross-check, never a reference answer.
Scores remain diagnostic model scores, not calibrated probabilities of error.
"""
from pathlib import Path
import hashlib
import json
import re
import difflib
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from scipy.optimize import linear_sum_assignment
try:
    from .facts import normal, direct_evidence
    from .terminology import translation_source
except ImportError:
    from facts import normal, direct_evidence
    from terminology import translation_source

MODEL_REVISIONS = dict(embedding='836121a0533e5664b21c7aacc5d22951f2b8b25b',nli='b5113eb38ab63efdd7f280f8c144ea8b13f978ce',translation='f8d333a098d19b4fd9a8b18f94170487ad3f821d')

MODEL_IDS = dict(embedding='sentence-transformers/LaBSE',
    nli='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7',
    translation='facebook/nllb-200-distilled-600M')


def sentences(text):
    # Keep semicolons and commas within propositions; preserve reorderings.
    return [x.strip() for x in re.split(r'(?<=[。！？])|(?<=[.!?])\s+(?=[A-Z])|\n+', text) if x.strip()]


def aligned_groups(zs, es, embeddings):
    if not zs or not es:return []
    sim=np.array([[float(embeddings[z]@embeddings[e]) for e in es] for z in zs])
    ii,jj=linear_sum_assignment(-sim)
    groups=[([int(i)],[int(j)],float(sim[i,j])) for i,j in zip(ii,jj)]
    # Preserve unmatched material by attaching it to the closest matched group.
    # The full-notice semantic pass decides meaning, not sentence-count parity.
    for i in sorted(set(range(len(zs)))-set(ii)):
        k=max(range(len(groups)),key=lambda k:max(sim[i,j] for j in groups[k][1]))
        groups[k][0].append(i)
    for j in sorted(set(range(len(es)))-set(jj)):
        k=max(range(len(groups)),key=lambda k:max(sim[i,j] for i in groups[k][0]))
        groups[k][1].append(j)
    return [(' '.join(zs[i] for i in sorted(iz)),' '.join(es[j] for j in sorted(ie)),score) for iz,ie,score in groups]


def digest(text):return hashlib.sha256(text.encode()).hexdigest()


class SignalExtractor:
    def __init__(self,cache=None,config=None,device=None,progress=print):
        self.progress=progress
        self.device=device or ('cuda' if torch.cuda.is_available() else 'cpu')
        torch.set_num_threads(4)
        torch.manual_seed(10)
        self.config=config or json.loads(Path(__file__).with_name('backbone_config.json').read_text())
        self.cache_path=Path(cache) if cache else None
        self.cache=json.loads(self.cache_path.read_text()) if self.cache_path and self.cache_path.exists() else {}
        self.progress('Loading V11 models (no V7 fallback).',flush=True)
        self.labse=SentenceTransformer(MODEL_IDS['embedding'],revision=MODEL_REVISIONS['embedding'],device=self.device)
        self.tok=AutoTokenizer.from_pretrained(MODEL_IDS['nli'],revision=MODEL_REVISIONS['nli'])
        self.nli=AutoModelForSequenceClassification.from_pretrained(MODEL_IDS['nli'],revision=MODEL_REVISIONS['nli']).to(self.device).eval()
        self.mt_tok=AutoTokenizer.from_pretrained(MODEL_IDS['translation'],revision=MODEL_REVISIONS['translation'],src_lang='zho_Hans')
        self.mt=AutoModelForSeq2SeqLM.from_pretrained(MODEL_IDS['translation'],revision=MODEL_REVISIONS['translation']).to(self.device).eval()
        labels=self.nli.config.id2label
        self.labels={name:i for i,name in labels.items()}
        assert all(x in self.labels for x in ('entailment','neutral','contradiction')),labels
        self.revisions=MODEL_REVISIONS.copy()
        self.progress('V11 ready on '+self.device,flush=True)

    def save_cache(self):
        if self.cache_path:
            self.cache_path.parent.mkdir(parents=True,exist_ok=True)
            tmp=self.cache_path.with_suffix('.tmp')
            tmp.write_text(json.dumps(self.cache,ensure_ascii=False))
            tmp.replace(self.cache_path)

    def translate(self,texts):
        missing=list(dict.fromkeys(t for t in texts if 'mt_nllb_hans_terms:'+digest(t) not in self.cache))
        for start in range(0,len(missing),8):
            batch=missing[start:start+8]
            inp=self.mt_tok([translation_source(t) for t in batch],return_tensors='pt',padding=True,truncation=False).to(self.device)
            if inp['input_ids'].shape[1]>512:raise ValueError('Translation segment exceeds token limit; no silent truncation.')
            with torch.inference_mode():
                out=self.mt.generate(**inp,max_new_tokens=300,num_beams=4,forced_bos_token_id=self.mt_tok.convert_tokens_to_ids("eng_Latn"))
            for t,ids,s in zip(batch,out,self.mt_tok.batch_decode(out,skip_special_tokens=True)):
                if len(ids)>=300 and self.mt.config.eos_token_id not in ids.tolist()[1:]:
                    self.cache['mt_nllb_hans_terms:'+digest(t)]={'error':'Translation output limit reached; the passage needs manual review.'}
                else:
                    self.cache['mt_nllb_hans_terms:'+digest(t)]=s
            self.save_cache()
            if start%40==0:self.progress(f'Translation cross-check: {min(start+8,len(missing))}/{len(missing)} new passages',flush=True)
        return {t:self.cache['mt_nllb_hans_terms:'+digest(t)] for t in texts}

    def nli_scores(self,pairs):
        def key(p):return 'nli:'+digest(json.dumps(p,ensure_ascii=False))
        missing=list(dict.fromkeys(p for p in pairs if key(p) not in self.cache))
        for start in range(0,len(missing),12):
            batch=missing[start:start+12]
            inp=self.tok([p[0] for p in batch],[p[1] for p in batch],return_tensors='pt',padding=True,truncation=False).to(self.device)
            if inp['input_ids'].shape[1]>512:raise ValueError('Inference segment exceeds token limit; no silent truncation.')
            with torch.inference_mode():scores=self.nli(**inp).logits.softmax(-1).cpu().tolist()
            for p,s in zip(batch,scores):self.cache[key(p)]={k:s[i] for k,i in self.labels.items()}
            self.save_cache()
            if start%120==0:self.progress(f'Contextual inference: {min(start+12,len(missing))}/{len(missing)} new pairs',flush=True)
        return {p:self.cache[key(p)] for p in pairs}

    def predict_many(self,inputs):
        # Only these fields cross the predictor boundary. References are not used.
        cases=[{k:r[k] for k in ('case_id','chinese','english')} for r in inputs]
        work=[]; invalid={}
        for i,r in enumerate(cases):
            z,e=normal(r['chinese']),normal(r['english'])
            if not z or not e:
                invalid[i]='Both language versions are required.';continue
            if max(len(z),len(e))>6000:
                invalid[i]='The 6,000-character input limit was exceeded.';continue
            # Short notices are compared as complete contexts. Long notices are
            # explicitly unresolved rather than silently dropping later text.
            if len(self.mt_tok(translation_source(z))['input_ids'])>480 or len(self.tok(z,e)['input_ids'])>480:
                invalid[i]='This notice exceeds V11’s validated context window; split it into shorter complete sections for review.';continue
            work.append((i,z,e))
        texts=list(dict.fromkeys([t for _,z,e in work for t in [z,e,*sentences(z),*sentences(e)]]))
        vectors=self.labse.encode(texts,normalize_embeddings=True,show_progress_bar=False,batch_size=16) if texts else []
        embeddings=dict(zip(texts,vectors))
        sentence_translations=self.translate(list(dict.fromkeys(t for _,z,_ in work for t in sentences(z))))
        valid=[]
        for i,z,e in work:
            if any(not isinstance(sentence_translations[t],str) for t in sentences(z)):
                invalid[i]='The translation cross-check could not cover this notice; manual review is needed.'
            else:valid.append((i,z,e))
        work=valid
        translations={z:' '.join(sentence_translations[t] for t in sentences(z)) for _,z,_ in work}
        valid=[]
        for i,z,e in work:
            if len(self.tok(translations[z],e)['input_ids'])>500:
                invalid[i]='The translated comparison exceeds the model context; manual review is needed.'
            else:valid.append((i,z,e))
        work=valid
        aligned={i:aligned_groups(sentences(z),sentences(e),embeddings) for i,z,e in work}
        quality_texts=list(dict.fromkeys([translations[z] for _,z,_ in work]+[v for v in sentence_translations.values() if isinstance(v,str)]+[a for i,z,e in work for a,b,sim in aligned[i]]+[' '.join(sentence_translations[t] for t in sentences(z) if t in a) for i,z,e in work for a,b,sim in aligned[i]]))
        quality_vectors=self.labse.encode(quality_texts,normalize_embeddings=True,show_progress_bar=False,batch_size=16) if quality_texts else []
        embeddings.update(dict(zip(quality_texts,quality_vectors)))
        pairs=[]
        for _,z,e in work:
            mt=translations[z]
            pairs.extend([(z,e),(e,z),(mt,e),(e,mt)])
            for a,b,sim in aligned[_]:
                am=' '.join(sentence_translations[t] for t in sentences(z) if t in a)
                pairs.extend([(a,b),(b,a),(am,b),(b,am)])
        scores=self.nli_scores(pairs)
        output={}
        for i,z,e in work:
            mt=translations[z]
            p=[scores[x] for x in [(z,e),(e,z),(mt,e),(e,mt)]]
            signal=dict(cross=[x['contradiction'] for x in p[:2]],cross_entailment=[x['entailment'] for x in p[:2]],cross_neutral=[x['neutral'] for x in p[:2]],
                        translated=[x['contradiction'] for x in p[2:]],
                        entailment=[x['entailment'] for x in p[2:]],
                        neutral=[x['neutral'] for x in p[2:]],
                        similarity=float(embeddings[z]@embeddings[e]),translation_quality=float(embeddings[z]@embeddings[mt]),translation=mt)
            facts=direct_evidence(z,e)
            groups=aligned[i]
            # Passage-level facts detect swapped roles even when document-level
            # value sets agree. Weak alignments do not establish a factual error.
            for a,b,sim in groups:
                if sim>=self.config['fact_alignment_min']:
                    facts.extend(direct_evidence(a,b))
            facts=list({json.dumps(x,sort_keys=True,ensure_ascii=False):x for x in facts}.values())
            reasons=[]
            for f in facts:
                reasons.append(dict(kind='explicit_fact',category=f['category'],
                    why=f"{f['category']} differs: English {f['english_value']} [{f['english_span']}]; Chinese {f['chinese_value']} [{f['chinese_span']}].",
                    **{k:v for k,v in f.items() if k!='category'}))
            passage_signals=[]
            for a,b,sim in groups:
                am=' '.join(sentence_translations[t] for t in sentences(z) if t in a)
                ps=[scores[x] for x in [(a,b),(b,a),(am,b),(b,am)]]
                ss=dict(cross=[x['contradiction'] for x in ps[:2]],cross_entailment=[x['entailment'] for x in ps[:2]],cross_neutral=[x['neutral'] for x in ps[:2]],entailment=[x['entailment'] for x in ps[2:]],neutral=[x['neutral'] for x in ps[2:]],translated=[x['contradiction'] for x in ps[2:]],similarity=sim,translation_quality=float(embeddings[a]@embeddings[am]),english_span=b,chinese_span=a,translation=am)
                passage_signals.append(ss)
            signal['passages']=passage_signals
            output[i]=dict(case_id=cases[i]['case_id'],flagged=bool(reasons),warnings=reasons,signals=signal,
                decision_path='v11_contextual_comparison',coverage='complete_notice',v7_fallback=False)
        for i,message in invalid.items():
            output[i]=dict(case_id=cases[i]['case_id'],flagged=True,warnings=[dict(kind='unresolved',category='input coverage',why=message,english_span=cases[i]['english'],chinese_span=cases[i]['chinese'])],decision_path='v11_unresolved',coverage='unresolved',v7_fallback=False)
        return [output[i] for i in range(len(cases))]

''')

### `features.py`: turning signals into 178 numbers
For the whole notice and for the smallest, largest and average passage, it lists every probability, its log-odds, the
minimum / maximum / difference between the two directions, similarity and translation quality. It adds the number of
passages and whether `facts.py` found a direct conflict. No case IDs, topics or answers are used.

In [ ]:
save('features.py', r'''"""Language-model signal features, without case IDs, topics or reference labels."""
import numpy as np
PROBS=('cross','cross_entailment','cross_neutral','translated','entailment','neutral')

def passage_vector(s):
    p=np.array([v for k in PROBS for v in s[k]],dtype=float)
    lp=np.log(np.clip(p,1e-5,1-1e-5)/np.clip(1-p,1e-5,1))
    summaries=[]
    for k in PROBS:
        a,b=s[k];summaries.extend([min(a,b),max(a,b),abs(a-b)])
    return np.r_[p,lp,summaries,[s['similarity'],s['translation_quality']]]

def vector(prediction):
    s=prediction['signals'];a=np.array([passage_vector(p) for p in s['passages']])
    direct=any(w['kind']=='explicit_fact' for w in prediction['warnings'])
    return np.r_[passage_vector(s),a.min(0),a.max(0),a.mean(0),len(a),int(direct)]


def feature_names():
    unit=[f'{p}.{direction}' for p in PROBS for direction in ['forward','reverse']]
    unit+=['logit.'+x for x in unit.copy()]
    unit+=[f'{p}.{stat}' for p in PROBS for stat in ['min','max','asymmetry']]
    unit+=['similarity','translation_quality']
    return [f'{group}.{name}' for group in ['whole','passage_min','passage_max','passage_mean'] for name in unit]+['passage_count','direct_fact']
''')

### `coverage_rules.py`: the missing-detail rule
Flags a condition word ("unless", "except", "and", 除非, 豁免, 及 ...) that appears in only one language, **but only when**
both NLI views strongly agree the other version lacks that detail (neutral ≥ 0.95 one way, entailment ≥ 0.85 the other)
and the translation quality is at least 0.7. This catches dropped conditions without flagging every "and".

In [ ]:
save('coverage_rules.py', r'''"""Corroborated one-way information loss with an explicit extra qualifier.

Operator presence is never enough to flag a pair. Both original bilingual NLI
and translated NLI must support the same direction of information loss.
"""
import re

OPERATORS = [
    ('exception', r'\b(?:unless|except|excluding|exempt)\b', r'除非|除.{0,18}[外以]|豁免|例外|不包括'),
    ('additional requirement', r'\b(?:and|along with|as well as|including|with)\b', r'及|和|與|並|連同|包括|附有'),
]


def unsupported_details(signal, english, chinese):
    found = []
    if signal['translation_quality'] < .7:
        return found
    for direction in (0, 1):
        if not (signal['cross_neutral'][direction] >= .95
                and signal['neutral'][direction] >= .95
                and signal['cross_entailment'][1-direction] >= .85
                and signal['entailment'][1-direction] >= .85):
            continue
        for category, en_pattern, zh_pattern in OPERATORS:
            em = re.search(en_pattern, english, re.I)
            zm = re.search(zh_pattern, chinese)
            if not ((direction == 0 and em and not zm) or
                    (direction == 1 and zm and not em)):
                continue
            match = em if direction == 0 else zm
            text = english if direction == 0 else chinese
            detail = re.split(r'[。！？]|(?<=[.!?])\s+', text[match.start():])[0]
            language = 'English' if direction == 0 else 'Chinese'
            found.append(dict(kind='unsupported_detail', category=category,
                why=f'{language} includes “{detail}”. Both semantic comparisons indicate that this detail is not supported by the other version; check for an omitted or added condition.',
                detail_language=language, unsupported_detail=detail,
                english_span=english, chinese_span=chinese,
                machine_backtranslation=signal['translation'],
                evidence_direction=direction))
    return found
''')

### `engine.py`: the decision
- `Decision.score()`: standardises the 178 numbers, multiplies them by the 178 learned weights in `learned_head.json`
  (logistic regression) and turns the result into a score from 0 to 1.
- `Decision.apply()`: a notice is flagged if the score is **0.785 or higher**, or if the missing-detail rule fires.
- `Decision.explain()`: when only the score fires, picks the passage that drives it and shows the English, the machine
  gloss of the Chinese and the words that differ, with a category such as "Exception or condition may differ".
- `Checker` and `check_notice()`: run everything and return a table with the status and the reason.

In [ ]:
save('engine.py', r'''"""Frozen V11: equivalence classification plus corroborated missing-detail checks.

The score is a learned decision score, not a calibrated probability of error.
No benchmark labels, scenario names or case IDs enter the scoring features.
"""
from pathlib import Path
import json
import re
import difflib
import numpy as np
try:
    from .backbone import SignalExtractor, MODEL_IDS, MODEL_REVISIONS
    from .features import vector, feature_names
    from .coverage_rules import unsupported_details
except ImportError:
    from backbone import SignalExtractor, MODEL_IDS, MODEL_REVISIONS
    from features import vector, feature_names
    from coverage_rules import unsupported_details


class Decision:
    def __init__(self):
        self.head = json.loads(Path(__file__).with_name('learned_head.json').read_text())
        assert self.head['feature_names'] == feature_names()

    def score(self, result):
        h = self.head
        x = (vector(result) - np.array(h['mean'])) / np.array(h['scale'])
        value = float(x @ np.array(h['coefficients']) + h['intercept'])
        return float(1 / (1 + np.exp(-np.clip(value, -50, 50))))

    def apply(self, result, english, chinese):
        if result['coverage'] != 'complete_notice':
            return result
        s = result['signals']
        score = self.score(result)
        detail_warnings = unsupported_details(s, english, chinese)
        for passage in s['passages']:
            detail_warnings += unsupported_details(passage, passage['english_span'], passage['chinese_span'])
        # Keep a detail once when both full-notice and aligned views support it.
        detail_warnings = list({(w['detail_language'], w['unsupported_detail']): w for w in detail_warnings}.values())
        flagged = score >= self.head['threshold'] or bool(detail_warnings)
        warnings = []
        if flagged:
            warnings = [w for w in result['warnings'] if w['kind'] == 'explicit_fact'] + detail_warnings
            if not warnings:
                warnings = [self.explain(result, english, chinese)]
        return dict(result, flagged=flagged, warnings=warnings, error_score=score,
            decision_threshold=self.head['threshold'],
            decision_path='v11_unsupported_detail' if detail_warnings else 'v11_learned_equivalence',
            v7_fallback=False)

    def explain(self, result, english, chinese):
        s = result['signals']
        options = []
        for p in s['passages']:
            ps = dict(p, passages=[p])
            options.append((self.score(dict(signals=ps, warnings=[])), p))
        # A passage is a navigation aid; the actual decision uses the full input.
        focus = max(options, key=lambda x: x[0])[1] if options else dict(s, english_span=english, chinese_span=chinese)
        e, z, mt = focus['english_span'], focus['chinese_span'], focus['translation']
        edits = []
        a, b = re.findall(r"\w+(?:['’-]\w+)*|[^\w\s]", e), re.findall(r"\w+(?:['’-]\w+)*|[^\w\s]", mt)
        for tag, i, j, k, l in difflib.SequenceMatcher(a=[x.lower() for x in a], b=[x.lower() for x in b], autojunk=False).get_opcodes():
            if tag != 'equal':
                edits.append(dict(english=' '.join(a[max(0,i-2):min(len(a),j+2)]), machine_backtranslation=' '.join(b[max(0,k-2):min(len(b),l+2)])))
        direction = int(np.argmax(np.array(focus['cross_neutral']) + np.array(focus['neutral'])))
        if min(focus['cross_neutral'][direction], focus['neutral'][direction]) >= .5:
            category = 'Missing or changed detail'
            if re.search(r'\b(?:unless|except|only if)\b|除非|豁免|例外', e+' '+z, re.I):
                category = 'Exception or condition may differ'
            elif re.search(r'\b(?:only|all|every|both|either|including|any)\b|僅|只|均|所有|包括', e+' '+z, re.I):
                category = 'Scope or requirement may differ'
        else:
            category = 'Facts or instructions may differ'
        # Always expose both actual passages; a generated gloss can be wrong.
        why = f'{category}. English states: “{e}”. The Chinese passage has the machine gloss: “{mt}”. Compare the highlighted wording against the original Chinese; the gloss can contain errors.'
        return dict(kind='equivalence_model', category=category, why=why,
            english_span=e, chinese_span=z, machine_backtranslation=mt,
            comparison_edits=edits)


class Checker(SignalExtractor):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.decision = Decision()

    def predict_many(self, inputs):
        results = super().predict_many(inputs)
        return [self.decision.apply(r, row['english'], row['chinese']) for r, row in zip(results, inputs)]

    def check_notice(self, chinese_text, english_text):
        import pandas as pd
        r = self.predict_many([dict(case_id='interactive', chinese=chinese_text, english=english_text)])[0]
        warnings = r['warnings'] or [dict(why='', category='')]
        return pd.DataFrame([dict(status='⚠️ Check' if r['flagged'] else '✅ No discrepancy found',
            chinese=chinese_text, english=english_text, coverage=r['coverage'],
            decision_path=r['decision_path'], **w) for w in warnings])
''')

### `learned_head.json`: the only trained part
The weights learned from 2,000 earlier cases: the 178 feature names, the mean and scale used to standardise each feature,
178 coefficients, the intercept, the decision threshold (0.785) and the regularisation strength (C = 0.01).

In [ ]:
save('learned_head.json', r'''{
  "version": "v11-equivalence-head",
  "feature_names": [
    "whole.cross.forward",
    "whole.cross.reverse",
    "whole.cross_entailment.forward",
    "whole.cross_entailment.reverse",
    "whole.cross_neutral.forward",
    "whole.cross_neutral.reverse",
    "whole.translated.forward",
    "whole.translated.reverse",
    "whole.entailment.forward",
    "whole.entailment.reverse",
    "whole.neutral.forward",
    "whole.neutral.reverse",
    "whole.logit.cross.forward",
    "whole.logit.cross.reverse",
    "whole.logit.cross_entailment.forward",
    "whole.logit.cross_entailment.reverse",
    "whole.logit.cross_neutral.forward",
    "whole.logit.cross_neutral.reverse",
    "whole.logit.translated.forward",
    "whole.logit.translated.reverse",
    "whole.logit.entailment.forward",
    "whole.logit.entailment.reverse",
    "whole.logit.neutral.forward",
    "whole.logit.neutral.reverse",
    "whole.cross.min",
    "whole.cross.max",
    "whole.cross.asymmetry",
    "whole.cross_entailment.min",
    "whole.cross_entailment.max",
    "whole.cross_entailment.asymmetry",
    "whole.cross_neutral.min",
    "whole.cross_neutral.max",
    "whole.cross_neutral.asymmetry",
    "whole.translated.min",
    "whole.translated.max",
    "whole.translated.asymmetry",
    "whole.entailment.min",
    "whole.entailment.max",
    "whole.entailment.asymmetry",
    "whole.neutral.min",
    "whole.neutral.max",
    "whole.neutral.asymmetry",
    "whole.similarity",
    "whole.translation_quality",
    "passage_min.cross.forward",
    "passage_min.cross.reverse",
    "passage_min.cross_entailment.forward",
    "passage_min.cross_entailment.reverse",
    "passage_min.cross_neutral.forward",
    "passage_min.cross_neutral.reverse",
    "passage_min.translated.forward",
    "passage_min.translated.reverse",
    "passage_min.entailment.forward",
    "passage_min.entailment.reverse",
    "passage_min.neutral.forward",
    "passage_min.neutral.reverse",
    "passage_min.logit.cross.forward",
    "passage_min.logit.cross.reverse",
    "passage_min.logit.cross_entailment.forward",
    "passage_min.logit.cross_entailment.reverse",
    "passage_min.logit.cross_neutral.forward",
    "passage_min.logit.cross_neutral.reverse",
    "passage_min.logit.translated.forward",
    "passage_min.logit.translated.reverse",
    "passage_min.logit.entailment.forward",
    "passage_min.logit.entailment.reverse",
    "passage_min.logit.neutral.forward",
    "passage_min.logit.neutral.reverse",
    "passage_min.cross.min",
    "passage_min.cross.max",
    "passage_min.cross.asymmetry",
    "passage_min.cross_entailment.min",
    "passage_min.cross_entailment.max",
    "passage_min.cross_entailment.asymmetry",
    "passage_min.cross_neutral.min",
    "passage_min.cross_neutral.max",
    "passage_min.cross_neutral.asymmetry",
    "passage_min.translated.min",
    "passage_min.translated.max",
    "passage_min.translated.asymmetry",
    "passage_min.entailment.min",
    "passage_min.entailment.max",
    "passage_min.entailment.asymmetry",
    "passage_min.neutral.min",
    "passage_min.neutral.max",
    "passage_min.neutral.asymmetry",
    "passage_min.similarity",
    "passage_min.translation_quality",
    "passage_max.cross.forward",
    "passage_max.cross.reverse",
    "passage_max.cross_entailment.forward",
    "passage_max.cross_entailment.reverse",
    "passage_max.cross_neutral.forward",
    "passage_max.cross_neutral.reverse",
    "passage_max.translated.forward",
    "passage_max.translated.reverse",
    "passage_max.entailment.forward",
    "passage_max.entailment.reverse",
    "passage_max.neutral.forward",
    "passage_max.neutral.reverse",
    "passage_max.logit.cross.forward",
    "passage_max.logit.cross.reverse",
    "passage_max.logit.cross_entailment.forward",
    "passage_max.logit.cross_entailment.reverse",
    "passage_max.logit.cross_neutral.forward",
    "passage_max.logit.cross_neutral.reverse",
    "passage_max.logit.translated.forward",
    "passage_max.logit.translated.reverse",
    "passage_max.logit.entailment.forward",
    "passage_max.logit.entailment.reverse",
    "passage_max.logit.neutral.forward",
    "passage_max.logit.neutral.reverse",
    "passage_max.cross.min",
    "passage_max.cross.max",
    "passage_max.cross.asymmetry",
    "passage_max.cross_entailment.min",
    "passage_max.cross_entailment.max",
    "passage_max.cross_entailment.asymmetry",
    "passage_max.cross_neutral.min",
    "passage_max.cross_neutral.max",
    "passage_max.cross_neutral.asymmetry",
    "passage_max.translated.min",
    "passage_max.translated.max",
    "passage_max.translated.asymmetry",
    "passage_max.entailment.min",
    "passage_max.entailment.max",
    "passage_max.entailment.asymmetry",
    "passage_max.neutral.min",
    "passage_max.neutral.max",
    "passage_max.neutral.asymmetry",
    "passage_max.similarity",
    "passage_max.translation_quality",
    "passage_mean.cross.forward",
    "passage_mean.cross.reverse",
    "passage_mean.cross_entailment.forward",
    "passage_mean.cross_entailment.reverse",
    "passage_mean.cross_neutral.forward",
    "passage_mean.cross_neutral.reverse",
    "passage_mean.translated.forward",
    "passage_mean.translated.reverse",
    "passage_mean.entailment.forward",
    "passage_mean.entailment.reverse",
    "passage_mean.neutral.forward",
    "passage_mean.neutral.reverse",
    "passage_mean.logit.cross.forward",
    "passage_mean.logit.cross.reverse",
    "passage_mean.logit.cross_entailment.forward",
    "passage_mean.logit.cross_entailment.reverse",
    "passage_mean.logit.cross_neutral.forward",
    "passage_mean.logit.cross_neutral.reverse",
    "passage_mean.logit.translated.forward",
    "passage_mean.logit.translated.reverse",
    "passage_mean.logit.entailment.forward",
    "passage_mean.logit.entailment.reverse",
    "passage_mean.logit.neutral.forward",
    "passage_mean.logit.neutral.reverse",
    "passage_mean.cross.min",
    "passage_mean.cross.max",
    "passage_mean.cross.asymmetry",
    "passage_mean.cross_entailment.min",
    "passage_mean.cross_entailment.max",
    "passage_mean.cross_entailment.asymmetry",
    "passage_mean.cross_neutral.min",
    "passage_mean.cross_neutral.max",
    "passage_mean.cross_neutral.asymmetry",
    "passage_mean.translated.min",
    "passage_mean.translated.max",
    "passage_mean.translated.asymmetry",
    "passage_mean.entailment.min",
    "passage_mean.entailment.max",
    "passage_mean.entailment.asymmetry",
    "passage_mean.neutral.min",
    "passage_mean.neutral.max",
    "passage_mean.neutral.asymmetry",
    "passage_mean.similarity",
    "passage_mean.translation_quality",
    "passage_count",
    "direct_fact"
  ],
  "mean": [
    0.24628928470611572,
    0.2252815707027912,
    0.6460645429491997,
    0.6283615529835224,
    0.10764917325973511,
    0.14635674357414247,
    0.26531446000933645,
    0.27815585389733316,
    0.515628787279129,
    0.4548352775573731,
    0.21905422127246857,
    0.26701358425617217,
    -3.041650003206985,
    -3.41684102220366,
    0.9992299059182815,
    0.9165383945340984,
    -3.4383767675830024,
    -3.178525005080174,
    -2.987313793018632,
    -2.8753345496760727,
    -0.18432057188310183,
    -0.7271711862426956,
    -2.5709520393026257,
    -2.14341401647621,
    0.2024720557630062,
    0.2690987996459007,
    0.06662674388289451,
    0.5652962021529675,
    0.7091298937797547,
    0.1438336916267872,
    0.061618919372558596,
    0.19238699746131896,
    0.13076807808876037,
    0.2191945226788521,
    0.3242757912278175,
    0.10508126854896545,
    0.36477980846166613,
    0.6056842563748359,
    0.24090444791316987,
    0.1220787216424942,
    0.36398908388614654,
    0.24191036224365234,
    0.7861164159178734,
    0.8006837535724044,
    0.10172061282396316,
    0.09598624432086944,
    0.5988612266778945,
    0.5632504693865776,
    0.08085197520256042,
    0.10121245431900025,
    0.13241613966226579,
    0.13884326222538948,
    0.38644464281201363,
    0.37105144569277765,
    0.15517488956451417,
    0.1622173651456833,
    -5.018170105532225,
    -5.230456646042185,
    0.5070477812575364,
    0.23842262597481018,
    -3.846469474193034,
    -3.7727514268868405,
    -4.96573868160921,
    -4.917183683615772,
    -1.3540579448801353,
    -1.7195519716141117,
    -3.262343286571926,
    -3.2354789589248267,
    0.08011914145946503,
    0.11769073575735092,
    0.03507623207569122,
    0.5024851906895638,
    0.6667747540473938,
    0.08637261402606965,
    0.04652522301673889,
    0.1381846034526825,
    0.08778060722351075,
    0.10537931832671166,
    0.16630333018302917,
    0.05621971014142037,
    0.2805952605307102,
    0.5079767778813838,
    0.1411159588098526,
    0.08368597400188446,
    0.24706702542304992,
    0.15048473346233368,
    0.7287560213804245,
    0.719392206911929,
    0.26004112988710404,
    0.24767220339179039,
    0.8096805330514908,
    0.7948522927463054,
    0.16191144251823425,
    0.21924657082557678,
    0.31096192994713784,
    0.3419140108823776,
    0.686610764682293,
    0.6631304840147495,
    0.36439016139507296,
    0.36979351818561557,
    -2.566729857328702,
    -2.822599793271983,
    2.6324323895722754,
    2.5275853158727073,
    -2.6397491625036302,
    -2.2991948696491478,
    -2.18925441093665,
    -1.973186838515811,
    1.449125622985429,
    1.2891736525103767,
    -1.2518840753455598,
    -1.1189831947365108,
    0.21613708648085594,
    0.2911808378100395,
    0.0801465747654438,
    0.7512131893336773,
    0.8497667647600174,
    0.1933609985411167,
    0.10526981282234192,
    0.27206641364097595,
    0.175575998544693,
    0.25881776443123816,
    0.3912297347784042,
    0.14722701945900918,
    0.5668357005417347,
    0.7570268793702125,
    0.3125804346501827,
    0.19790985536575317,
    0.49962124252319334,
    0.33270557928085326,
    0.8209226298332214,
    0.8337574018239975,
    0.1692365838428339,
    0.16038391205966482,
    0.715807847828667,
    0.6908990433404842,
    0.11496121875246365,
    0.14870836847424515,
    0.2081124401470027,
    0.2244936335921287,
    0.5447477107882495,
    0.5183039322127899,
    0.24713782036503157,
    0.25720445786714535,
    -3.8493642346349852,
    -4.093931064301562,
    1.6147804990876635,
    1.4421788086922502,
    -3.250574101735139,
    -3.0715068044128113,
    -3.6491301320462144,
    -3.536576079214378,
    0.10813267858978909,
    -0.1828317324565221,
    -2.292531767908219,
    -2.193743916529805,
    0.13802574961880854,
    0.19159474628369014,
    0.0535689966648817,
    0.6372769230465096,
    0.7694299681226405,
    0.1321530450761318,
    0.0697103570262591,
    0.1939592302004499,
    0.12424887317419046,
    0.17001463626921207,
    0.2625914374699196,
    0.09257680120070771,
    0.41901414748926935,
    0.6440374955117706,
    0.22502334802250057,
    0.13324086440205563,
    0.3711014138301214,
    0.2378605494280655,
    0.7721307734946417,
    0.7807683269051217,
    1.805,
    0.0525
  ],
  "scale": [
    0.3788369592975607,
    0.37107471971227235,
    0.40569282510705107,
    0.41094053570732914,
    0.2096094793639669,
    0.2579130523477987,
    0.38779701061567234,
    0.38897488824764287,
    0.4270168317224434,
    0.4233504755203603,
    0.31958469825113595,
    0.3444760347287676,
    4.137739535667919,
    4.112723128620906,
    3.9070594301837587,
    3.9794203807316446,
    2.205566120947592,
    2.576563387212927,
    4.407132840652314,
    4.4484997118878775,
    4.316931359431509,
    4.304627954305373,
    3.044528980779519,
    3.1947896233061637,
    0.3547414662454877,
    0.3916292660016211,
    0.14926841368954477,
    0.4127900778641767,
    0.3909933200849113,
    0.2269536472612704,
    0.15057684366999385,
    0.2828089168277447,
    0.22328012596160562,
    0.362672877726715,
    0.40585438655964395,
    0.19467125907116095,
    0.40354860301324347,
    0.41418350233153767,
    0.30335677676987166,
    0.22669818120855673,
    0.37589332404810155,
    0.304808049567732,
    0.0872736922459864,
    0.11005064635548224,
    0.25515364157038906,
    0.2571182390590314,
    0.3951022099569551,
    0.4152559812575754,
    0.19030133512808725,
    0.22781531580323128,
    0.29587042499829214,
    0.30017931660782116,
    0.4130291872920726,
    0.4164055423992853,
    0.2839272684831014,
    0.2898183388734324,
    3.1865282362336242,
    3.1681948482924978,
    3.6770102570086487,
    3.874832915695073,
    2.0406514308191546,
    2.3896084186224993,
    3.6967972139276,
    3.821884791623272,
    4.130604109712602,
    4.348269865035643,
    2.8344622494718497,
    2.9146422838644312,
    0.23185696169736844,
    0.2770441275252423,
    0.11308480058480215,
    0.4043773820297292,
    0.39070915737311357,
    0.18683767392683756,
    0.14012697535094473,
    0.2545939053833102,
    0.1926832969933711,
    0.26701702455576726,
    0.3232441976674279,
    0.15019100458090964,
    0.38129595972082403,
    0.4253088164131201,
    0.25128574825395605,
    0.1980188344946817,
    0.3407957559910621,
    0.25304805996648455,
    0.10128669142673608,
    0.2128726745212775,
    0.3640975738235432,
    0.3719885858154599,
    0.32421795905049994,
    0.34224355217962815,
    0.23076048603882532,
    0.30850975130809816,
    0.39638598521342455,
    0.4089463197724776,
    0.40513489692601506,
    0.41273533679091473,
    0.36755781187396236,
    0.37893789643830356,
    3.780561440504799,
    3.905454181256465,
    3.1706433811652923,
    3.442381509276941,
    2.13744363138162,
    2.674576566341037,
    4.256264560606601,
    4.460330764360177,
    4.108478901988016,
    4.2300924504332995,
    3.047143094129245,
    3.3230136025332864,
    0.3480930951085185,
    0.3836311934226638,
    0.15535429877973914,
    0.35492894235077255,
    0.3027556840986371,
    0.2506770537892342,
    0.18417899237367974,
    0.3192560342080583,
    0.24632195410803456,
    0.37417609066411794,
    0.41921024779546706,
    0.2320848775676233,
    0.42669346070201225,
    0.37254873536082217,
    0.33593164055909436,
    0.2720596198375316,
    0.3936448040498482,
    0.3292932129176955,
    0.11207814482529788,
    0.05907169414076359,
    0.27028405268103456,
    0.2728263227715932,
    0.3196794688451266,
    0.3346073455867121,
    0.19247526437838017,
    0.23475663170073646,
    0.3026866827774332,
    0.30685544112246155,
    0.36076800010720284,
    0.3685542476143115,
    0.28390485642288954,
    0.2939093017600302,
    3.081008208447341,
    3.140146967077327,
    3.0369738566182285,
    3.2735559295160637,
    1.8992103079666485,
    2.262158563328244,
    3.487320385660336,
    3.6130901252109546,
    3.678253105002573,
    3.789015510076084,
    2.618155117859623,
    2.7299041095899326,
    0.25073109696432005,
    0.2887335561295071,
    0.11797289337531744,
    0.33667722986301,
    0.30726115652221775,
    0.19277014101357212,
    0.14501887813339578,
    0.2550841866265877,
    0.19688313448034453,
    0.2793774402196232,
    0.3228234147357445,
    0.1596972456594498,
    0.3569096626276318,
    0.35416043048297513,
    0.26229342070417155,
    0.20768726253534034,
    0.3250786413271245,
    0.25715142786239387,
    0.08768027740370513,
    0.10594196963049031,
    0.9486701218020939,
    0.22303306929691014
  ],
  "coefficients": [
    0.02336571903981321,
    0.054959473191997656,
    -0.04906775262951636,
    -0.07480003924917251,
    0.05268657177545898,
    0.0401215123799222,
    0.053726737361079406,
    -0.017205343423578334,
    -0.09099435792145182,
    0.04616402846213692,
    0.05639014790397074,
    -0.03733579271684519,
    0.022141128701483256,
    0.1060773802394267,
    -0.0780118584678635,
    -0.12131491681481811,
    0.07074805880138382,
    -0.021943551589014888,
    0.06088491106157337,
    0.07118081502560818,
    -0.10546850322486859,
    -0.026846184661596852,
    0.004288355425815176,
    0.03529768788994775,
    0.03141007764509859,
    0.04622589177060327,
    0.04663381149184418,
    -0.08741437444513522,
    -0.03724108879683355,
    0.0948333271622975,
    0.046273003029490355,
    0.05100201231460681,
    0.03339384140574456,
    0.004301185512780693,
    0.031002987013189473,
    0.05662250812365006,
    -0.031088163128438703,
    -0.016338106621903256,
    0.01904889233111476,
    0.01191567711232879,
    0.00654143100483841,
    -0.0007951958130275291,
    -0.276396607238605,
    0.08452826809185367,
    -0.025456858475829996,
    0.03780722663267736,
    -0.09624943812973911,
    -0.08325031598988498,
    -0.015443924281277098,
    0.03894609762412518,
    -0.10287763289699005,
    -0.033060270487786834,
    -0.07194978790888606,
    -0.0693967196102066,
    0.0273488987542896,
    0.0011036326414267778,
    -0.049911647155950374,
    0.01751062508432909,
    -0.15828606993120173,
    -0.13419480475190443,
    0.02610358756376992,
    -0.03660746113021357,
    -0.029850738751827307,
    0.000634169598833633,
    -0.14056508777208274,
    -0.04055733266748282,
    0.013033350380375296,
    0.042056498891614,
    0.025430549620481427,
    -0.009635387317764274,
    -0.04243908521879972,
    -0.10382452226591107,
    -0.07375467527743852,
    0.03743448946195423,
    0.0296967386977625,
    0.027729834078963955,
    -0.00044281896455697654,
    -0.08300791783648973,
    -0.049148194183113995,
    0.006827670972680142,
    -0.1068925593666277,
    -0.09139971243225335,
    0.04535473204023727,
    -0.00884946005307738,
    0.040813579682532033,
    0.013406665704234974,
    -0.22657412357901519,
    0.10088912146815435,
    0.027276247827412895,
    0.08859239480300532,
    0.05352227554993485,
    -0.022037296441835615,
    0.08382825321126783,
    -0.06560851046963057,
    0.13074049002886795,
    0.08511065616713734,
    0.00040153808922093473,
    -0.01085645094364142,
    -0.03927091293066299,
    0.013292989828801356,
    0.034125161359560584,
    0.11653330643256138,
    0.01398154892640771,
    0.007672523120267042,
    0.06957423021343428,
    -0.10092590856705394,
    0.088956417974446,
    0.02407700254566332,
    0.0436517670561438,
    0.016676635083997433,
    0.016875062191776907,
    0.030543382221218066,
    0.04618738294983853,
    0.07272294312141556,
    0.06094738014659067,
    -0.003883359027825405,
    0.015020271927969937,
    0.026266812008932487,
    -0.01202248156366658,
    -0.03863784194176175,
    -0.0738357415590142,
    0.10045552107868905,
    0.10786800976420696,
    0.02916504813402874,
    -0.008830627743085382,
    0.030226963255811697,
    0.030387874669377587,
    0.01670897360045151,
    -0.009592987810167973,
    -0.003236085725949579,
    -0.1926909176955715,
    0.16177944267099473,
    -0.0018983371079837412,
    0.05034497376890043,
    -0.01956888835931615,
    -0.03717011820873982,
    0.03514808803889868,
    -0.005467413842220863,
    0.03604356341553141,
    0.052821855532936036,
    -0.02714304938639197,
    -0.05037966746731012,
    -0.003926460284801309,
    0.008045207220760438,
    -0.0021739472539904756,
    0.0685906491287149,
    -0.06040849655744752,
    -0.05662887333125279,
    0.039106772437409974,
    -0.062170634561218484,
    0.03860100013751061,
    0.029461565023623747,
    -0.04104007909066295,
    -0.030666549844980778,
    0.010673701220498287,
    0.03404061260255052,
    0.014938546892493876,
    0.03282190575328281,
    0.04858088289359578,
    -0.042312658077754466,
    -0.014474520522368218,
    0.05082867368583709,
    0.015103456860349064,
    0.012902953937172196,
    0.005592419803890623,
    0.0363818049868869,
    0.05251888216430546,
    0.042518387230415546,
    -0.04832567461954623,
    -0.03137584806564826,
    0.023392948060015732,
    0.00971698488359783,
    -0.0023633478166190966,
    -0.010835513965229512,
    -0.2213771291724383,
    0.13467147024351153,
    -0.06809381019709089,
    0.4550033577800869
  ],
  "intercept": -2.21657763051741,
  "threshold": 0.785,
  "C": 0.01
}
''')

### `run.py`: the benchmark runner
Reads a CSV with only `case_id`, `english` and `chinese` (no answers), refuses to run if any checker file has changed
(`--require-frozen`), and saves every prediction with the file checksums, model versions and package versions.

In [ ]:
save('run.py', r'''"""Run the frozen predictor on an inputs-only CSV; never score during inference."""
import argparse,csv,json,hashlib,sys,time,platform,importlib.metadata,io
from pathlib import Path
from datetime import datetime,timezone
from engine import Checker,MODEL_IDS
p=argparse.ArgumentParser()
p.add_argument('--inputs',required=True);p.add_argument('--output',required=True);p.add_argument('--cache',required=True)
p.add_argument('--batch-size',type=int,default=50);p.add_argument('--require-frozen',action='store_true')
a=p.parse_args();root=Path(__file__).resolve().parent
source_files=['__init__.py','engine.py','backbone.py','facts.py','terminology.py','backbone_config.json','features.py','coverage_rules.py','learned_head.json','run.py']
hashes={f:hashlib.sha256((root/f).read_bytes()).hexdigest() for f in source_files}
if a.require_frozen:
 manifest=json.loads((root/'frozen_manifest.json').read_text());assert hashes==manifest['files'],'Frozen checker changed'
raw=Path(a.inputs).read_bytes()
rows=list(csv.DictReader(io.StringIO(raw.decode('utf-8-sig'))))
assert set(rows[0])=={'case_id','english','chinese'},'Prediction input must contain no gold labels or rationales'
assert len({r['case_id'] for r in rows})==len(rows)
started=datetime.now(timezone.utc).isoformat()
checker=Checker(cache=a.cache)
results=[];output=Path(a.output);output.parent.mkdir(parents=True,exist_ok=True)
partial=output.with_suffix('.jsonl');partial.write_text('')
for i in range(0,len(rows),a.batch_size):
 batch=checker.predict_many(rows[i:i+a.batch_size]);results.extend(batch)
 with partial.open('a') as f:
  for r in batch:f.write(json.dumps(r,ensure_ascii=False)+'\n')
 print('PREDICTIONS',len(results),'/',len(rows),flush=True)
metadata=dict(version='V11',started=started,completed=datetime.now(timezone.utc).isoformat(),
 inputs_sha256=hashlib.sha256(raw).hexdigest(),files=hashes,model_ids=MODEL_IDS,
 model_revisions=checker.revisions,device=checker.device,python=platform.python_version(),
 packages={p:importlib.metadata.version(p) for p in ['torch','transformers','sentence-transformers','sentencepiece','pandas','scipy','numpy','opencc-python-reimplemented']},
 predictions=len(results),v7_fallback_calls=0)
output.write_text(json.dumps(dict(metadata=metadata,results=results),ensure_ascii=False,indent=2))
print('V11 COMPLETE',len(results),flush=True)
''')

### Small settings files
`backbone_config.json` (minimum similarity for matching facts), `__init__.py` (marks the folder as a package) and
`frozen_manifest.json` (the SHA-256 checksum of every file, the model versions and how V11 was trained).

In [ ]:
save('backbone_config.json', r'''{
  "fact_alignment_min": 0.6
}
''')

In [ ]:
save('__init__.py', r'''"""V11 bilingual notice checker."""
''')

In [ ]:
save('frozen_manifest.json', r'''{
  "version": "V11",
  "frozen_at": "2026-11-15T03:38:28.344024+00:00",
  "files": {
    "__init__.py": "444fda88bfdca23162f1335ed9a71c63eda802c74fa8430a8eb16993925ad688",
    "engine.py": "508f6b9964c7cb3b32226a2e6202585c9c0f57243f3c8e53006102671090b75d",
    "backbone.py": "34d41f209a06dedf8d43efee7ec5694d890b821e3192254bd208d28a6c08007c",
    "facts.py": "ccbf3a05781feda7974b2b796003212bc41ce4e5fe0bed503eb348254a29e78f",
    "terminology.py": "3f11a58169de2a81fcaf386479f3ff8641d393e88441a7812a3edce1fe1de190",
    "backbone_config.json": "fd2d06ad164b79c2fd2e33e3538b0877c8ad1d730249297193e1175a6ab00d6b",
    "features.py": "d2c6df6f1c4d4ea20b446e8422283e45e4eccdbc858edeaf17beb107bba4b012",
    "coverage_rules.py": "06e206fdb6af8f454c64a11695d4543e7170adfb127fa741b071e85f4045a486",
    "learned_head.json": "8e90070a927632019816827695a5df6c0332a8708411f9bcd71616fbe7c1591f",
    "run.py": "77642754436ac9c6a0ff2157a44bf319966aae97f7c363f9e7ff94bbf7eb55c0"
  },
  "source_bundle_sha256": "d784941bcdc91bb9ef407ed9c5f2b52bf1cb4cfaf673b0772304ea3d18b653ea",
  "model_revisions": {
    "embedding": "836121a0533e5664b21c7aacc5d22951f2b8b25b",
    "nli": "b5113eb38ab63efdd7f280f8c144ea8b13f978ce",
    "translation": "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
  },
  "model_ids": {
    "embedding": "sentence-transformers/LaBSE",
    "nli": "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    "translation": "facebook/nllb-200-distilled-600M"
  },
  "development_data": [
    "holdout_v9/v9_unseen_1000.csv",
    "holdout_v10/v10_blind_1000.csv"
  ],
  "training_records_sha256": "8e53512a7d4b6f3f51fd7ec60f987791f6262c3f77a39fd590dade5dcca08fb5",
  "training_procedure": "regularized logistic head on 2000 old cases; 5 family-disjoint folds for model and threshold selection; extra corroborated qualifier-loss rule",
  "decision_threshold": 0.785,
  "C": 0.01,
  "holdout_text_or_labels_read": false,
  "no_further_tuning": true,
  "smoke_cases": 12,
  "python": "3.11.14",
  "packages": {
    "torch": "2.15.0",
    "transformers": "5.22.1",
    "sentence-transformers": "5.7.0",
    "sentencepiece": "0.2.3",
    "pandas": "3.1.1",
    "scipy": "1.17.1",
    "numpy": "2.4.6",
    "scikit-learn": "1.9.1",
    "opencc-python-reimplemented": "0.1.7"
  }
}
''')

### Check the files and load V11
Every file is compared with the checksum recorded when V11 was frozen. If all match, the checker is loaded.

In [ ]:
import json, hashlib, sys
manifest = json.loads((folder / 'frozen_manifest.json').read_text())
for name, digest in manifest['files'].items():
    assert hashlib.sha256((folder / name).read_bytes()).hexdigest() == digest, f'{name} differs from the frozen version'
print('All', len(manifest['files']), 'files match their frozen checksums')
sys.path.insert(0, '/content')
sys.path.insert(0, str(folder))  # V11's modules import each other by plain name
from v11_checker.engine import Checker
checker = Checker(cache='/content/v11_cache.json')
check_notice = checker.check_notice
print('Frozen V11 loaded. No V7 fallback.')

In [ ]:
chinese_text='請於星期五提交表格。'
english_text='Please submit the form on Friday.'
check_notice(chinese_text, english_text)


## Demo app
Paste the Traditional Chinese and English versions of a notice. V11 checks them and shows each warning with its reason.
A person always makes the final call. The first example is a correct translation; the second changes the deadline.

In [ ]:
import gradio as gr

EXAMPLES = [
    ["報名截止日期為11月20日。活動將於11月27日舉行。參加者毋須繳付任何費用。",
     "The application deadline is 20 November. The activity will be held on 27 November. Participants do not need to pay any fee."],
    ["報名截止日期為11月20日。活動將於11月27日舉行。參加者毋須繳付任何費用。",
     "The application deadline is 27 November. The activity will be held on 20 November. Participants do not need to pay any fee."],
    ["講座以廣東話進行。時間：下午二時三十分至四時三十分。", "The lecture will be conducted in English. Time: 2.30pm to 4.30pm."],
]

def demo_check(chinese_text, english_text):
    result = check_notice(chinese_text, english_text)
    columns = [c for c in ["status", "category", "why"] if c in result.columns]
    return result[columns]

gr.Interface(
    fn=demo_check,
    inputs=[gr.Textbox(label="Traditional Chinese version", lines=6),
            gr.Textbox(label="English version", lines=6)],
    outputs=gr.Dataframe(label="V11 findings", wrap=True),
    examples=EXAMPLES,
    title="Public Notice Check (V11)",
    description="A last check before a bilingual notice goes out. It flags passages to review; a person makes the final call.",
).launch()

In [ ]:
# Optional: rerun the frozen benchmark. Set RUN_BENCHMARK = True to use it.
# It frees the demo's models first, so run it after you have finished with the demo.
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    # Upload the INPUTS-ONLY CSV supplied in the benchmark bundle.
    # Its three columns are case_id, english, chinese; keep the answer key separate.
    from google.colab import files
    import csv, io, subprocess, gc, torch
    uploaded=files.upload()
    assert len(uploaded)==1, 'Upload one inputs-only CSV.'
    name, raw=next(iter(uploaded.items()))
    rows=list(csv.DictReader(io.StringIO(raw.decode('utf-8-sig'))))
    assert rows and set(rows[0])=={'case_id','english','chinese'}
    input_path=Path('/content/v11_inputs.csv')
    input_path.write_bytes(raw)
    # Release interactive model memory before starting the frozen benchmark runner.
    del check_notice, checker
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    subprocess.run([sys.executable,str(folder/'run.py'),'--inputs',str(input_path),
        '--output','/content/v11_predictions.json','--cache','/content/v11_cache.json',
        '--require-frozen'],check=True)
    files.download('/content/v11_predictions.json')
else:
    print('Benchmark skipped (set RUN_BENCHMARK = True to run it).')